<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/01%20-%20Basics/01b_pandas.ipynb)

# 01b · 文件读写与 Pandas 表格处理

在上一节（`01a`）的最后，我们用“列表套字典”拼出了一张包含三个分子的小表。但在真实的科研工作里，数据往往来自仪器导出的文本、文献附件里的 Excel，或是包含上万个化合物的 CSV 数据库。在实际做机器学习项目时，大约八成的时间都花在**数据的读取、清洗、筛选和整理**上。

这一节我们分两步走：
1. 先看怎样用纯 Python 读写最基础的文本文件（理解底层文件操作）；
2. 重点掌握 **[Pandas](https://pandas.pydata.org/)**——Python 处理表格数据的标准工具，用它来探索一个包含近万个化合物的真实水溶解度数据集（**AqSolDB**）。

### 本节内容
1. 纯 Python 读写文本文件
2. Pandas 入门：从字典列表到 DataFrame，读入 AqSolDB 数据集
3. 数据初探：查看表格结构与统计概览
4. 选列、选行与条件筛选
5. 排序与极值查找
6. 特征计算（加列、`apply`）与分组统计（`groupby`）
7. 多表合并（`merge` / `concat`）、缺失值处理与保存

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "01 - Basics"))


---
## 1. 纯 Python 读写文本文件

在调用大型工具库之前，我们先看看 Python 自带的文件读写是怎么工作的。假设你手头有一份常用有机半导体分子的名单，想把它存成硬盘上的一个 `.txt` 文本文件，每行写一个名字：

In [ ]:
osc_names = ["pentacene", "rubrene", "C8-BTBT", "TIPS-pentacene", "P3HT"]

with open("osc_names.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(osc_names))

上面这三行代码是 Python 读写文件的标准写法：
- `open("osc_names.txt", "w")`：打开文件。第二个参数 `"w"`（write）表示**写入模式**（如果文件不存在会自动新建；如果已存在，则会清空覆盖原有内容。想在文件末尾追加内容可以用 `"a"`，只读模式用 `"r"`）。
- `with ... as f:`：这是一个安全管家——不管代码块里有没有报错，出了这个缩进块之后，Python 都会自动把文件关好，避免文件被锁死或数据没写进硬盘。
- `"\n".join(osc_names)`：用换行符 `\n` 作粘合剂，把列表里的 5 个字符串连成一段带换行的长文本：

In [ ]:
print("\n".join(osc_names))

现在我们把刚才存好的 `osc_names.txt` 重新读回内存里。用 `f.readlines()` 可以把文件的每一行读成列表里的一个元素：

In [ ]:
with open("osc_names.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
print(lines)

仔细看打印出来的列表：除了最后一个名字，前面每个名字末尾都带着一个尾巴 `\n`（换行符）。这就是为什么我们在 `01a` 里专门介绍了字符串的 `.strip()` 方法——配合列表推导式，一行就能把首尾的换行符和空格清理干净：

In [ ]:
with open("osc_names.txt", "r", encoding="utf-8") as f:
    names = [line.strip() for line in f]
print(names)

平时如果只是存几行日志、或者解析课题组自编程序输出的特殊文本（比如 Gaussian / VASP 的输出文件），`with open(...)` 非常管用。但如果面对的是规整的行列式表格数据，手写文件解析就太费劲了——这时候就该 **Pandas** 上场了。

---
## 2. Pandas 入门与数据集读入

可以把 **Pandas** 理解成“写代码版的 Excel”。它最重要的两个概念是：
- **`DataFrame`**：整张二维表格（有行索引 Index，也有列名 Columns）。
- **`Series`**：表格里的单独一列（或单独一行），底层其实就是带了标签的 NumPy 数组。

还记得 `01a` 里我们写的那个“列表套字典” `molecules` 吗？把它直接传给 `pd.DataFrame(...)`，瞬间就变成了一张漂亮的表格：

In [ ]:
import pandas as pd

molecules = [
    {"name": "pentacene", "homo": -5.0, "lumo": -3.2},
    {"name": "rubrene",   "homo": -5.4, "lumo": -3.1},
    {"name": "C60",       "homo": -6.1, "lumo": -4.0},
]

mini_df = pd.DataFrame(molecules)
mini_df

当然，实际研究中的数据表通常存在 CSV（逗号分隔值）或 Excel 文件里，常用的读写命令只有四行：

```python
df = pd.read_csv("file.csv")          # 读取 CSV 文件
df = pd.read_excel("file.xlsx")       # 读取 Excel 文件
df.to_csv("out.csv", index=False)     # 保存为 CSV（不保存左侧行号）
df.to_excel("out.xlsx", index=False)  # 保存为 Excel
```

### 本章主线数据集：AqSolDB 水溶解度数据库

接下来几节课，我们会反复使用 **AqSolDB**（Sorkun 等, *Scientific Data*, 2019）数据集。它收集了 **9482 个化合物**在室温下的实验水溶解度，并预先用 RDKit 算好了 17 个分子描述符。主要字段包括：

| 列名 | 物理化学含义 |
|---|---|
| `Name` | 化合物名称 |
| `SMILES` | 分子结构的字符串表示（`01d` 详述） |
| `Solubility` | 实验水溶解度 $\log S$，单位 $\log_{10}(\text{mol/L})$。**数值越大越易溶**（例如 $0$ 代表 $1\text{ mol/L}$，$-6$ 代表 $1\text{ }\mu\text{mol/L}$） |
| `MolWt` | 相对分子质量 (g/mol) |
| `MolLogP` | 辛醇/水分配系数的计算值（$\log P$），数值越大代表分子越疏水（“亲油”） |
| `NumAromaticRings` | 芳香环个数 |
| `NumHeteroatoms` | 杂原子（非碳、氢原子）个数 |
| `TPSA` | 拓扑极性表面积 (Å²)，反映极性基团暴露的面积 |

> **为什么做有机光电材料也要关心“溶解度”？**
> 大面积柔性有机电子器件（如旋涂、刮涂、喷墨打印制备的 OFET 和 OPV）依赖**溶液加工**。然而，为了获得高迁移率，有机半导体需要大尺寸的刚性 $\pi$ 共轭稠环骨架，这会导致极强的分子间 $\pi\text{--}\pi$ 堆积，使分子像砖头一样难溶。因此，化学家必须在共轭核上引入柔性侧链（如 TIPS-并五苯的硅烷基、C8-BTBT 的辛基、P3HT 的己基）来改善溶解度。在这份数据里，我们会亲眼验证“芳香环越多、分子量越大，溶解度怎样急剧下降”。

运行下面的代码读入 `data/solubility.csv`：

In [ ]:
import os
import pandas as pd

# 本地运行时直接读取 data/solubility.csv；如果在 Colab 上找不到文件，会弹出上传按钮
DATA_FILE = "data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("请上传 solubility.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 solubility.csv，请确保它在 data/ 目录下")

df = pd.read_csv(DATA_FILE)
print(f"读取成功！共包含 {len(df)} 个化合物。")

---
## 3. 数据初探：查看表格结构与统计概览

拿到一份陌生的数据集，千万不要上来就往模型里塞，第一步永远是先摸清它的长相：有几行几列？每列叫什么名字？有没有缺失值？数值范围大概在什么区间？

In [ ]:
df.head()          # 查看前 5 行（括号里填数字比如 df.head(10) 可以看前 10 行；看末尾用 df.tail()）

In [ ]:
print("表格尺寸 (行数, 列数):", df.shape)   # 注意 shape 是属性，不带括号
print("所有列名:", list(df.columns))

In [ ]:
df.info()          # 检查每一列的数据类型 (float64 / int64 / object) 以及非空值数量

`df.describe()` 是最省事的体检工具，它会一口气算出所有数值列的均值（`mean`）、标准差（`std`）、最小值（`min`）、四分位数（`25%`, `50%`, `75%`）和最大值（`max`）：

In [ ]:
df.describe().round(2)

### 🏋️ 练习 01b-1

观察上面 `df.describe()` 输出的表格，回答两个问题（直接双击下面那格文字写下你的结论即可）：
1. 数据集中水溶解度 `Solubility`（$\log S$）的**平均值（`mean`）**和**中位数（`50%`）**分别是多少？
2. 相对分子质量 `MolWt` 的**最大值（`max`）**是多少？它看起来像一个普通的小分子吗？这给你什么提醒？

*双击这里写下你的观察*

---
## 4. 选列、选行与条件筛选

### 4.1 取出单独一列或多列

从 `DataFrame` 里按列名取数据，语法和字典非常像：
- 传**一个列名字符串** `df["Solubility"]`，取出来的是单列（`Series`）；
- 传**一个包含多个列名的列表** `df[["Name", "Solubility", "MolWt"]]`（注意是两层方括号！外层表示索引，内层是一个列表），取出来的是一张只包含这几列的新 `DataFrame`。

In [ ]:
df["Solubility"].head()

In [ ]:
df[["Name", "Solubility", "MolWt"]].head()

取出来的单列（`Series`）自带丰富的统计方法。除了 `.mean()`、`.max()`、`.min()`，对于离散的整数或类别列，`.value_counts()` 可以直接统计每个值出现了多少次：

In [ ]:
print("平均 logS:", round(df["Solubility"].mean(), 2))
print("最大 logS:", round(df["Solubility"].max(), 2))
print("\n不同芳香环数的分子数量分布（前 6 项）:")
print(df["NumAromaticRings"].value_counts().head(6))

### 4.2 按位置或标签选行（`iloc` 与 `loc`）

想取表格里的某几行时，Pandas 提供了两套索引器：
- **`df.iloc[...]`**（integer location）：纯粹按**第几行、第几列的整数位置**（从 0 开始）来取，规则和 Python 列表切片完全一致。
- **`df.loc[行标签, 列名]`**（label location）：按**行索引标签（Index）和列名**来取。

In [ ]:
df.iloc[0]          # 取第 0 行的全部字段

In [ ]:
df.iloc[10:13]      # 切片：取第 10、11、12 行（左闭右开，不含 13）

In [ ]:
# 用 .loc 同时指定行标签和列名：
print("第 0 行的分子名称:", df.loc[0, "Name"])

# 配合 .idxmin()（返回最小值所在的行标签）找出水溶解度最低的那个分子：
min_idx = df["Solubility"].idxmin()
df.loc[min_idx, ["Name", "SMILES", "Solubility", "MolWt"]]

### 4.3 按条件筛选行（布尔索引）

在实际数据分析中，我们很少靠背行号去取数据，绝大多数时候都是**按条件筛选**（比如“挑出所有极易溶于水的分子”或“挑出所有不含杂原子的多环芳烃”）。

它的原理和 `01a` 最后学的 NumPy 数组筛选一模一样：先写出一个比较条件 `df["Solubility"] > 0`，这会产生一列由 `True` 和 `False` 组成的掩码（Mask）；再把这列掩码塞进 `df[...]`，就只留下结果为 `True` 的那些行：

In [ ]:
mask = df["Solubility"] > 0
print("logS > 0 的分子共有:", mask.sum(), "个")    # 求和时 True 算 1，False 算 0

# 把条件放进 df[...] 中筛选出对应行：
df[df["Solubility"] > 0][["Name", "SMILES", "Solubility"]].head()

> **多条件组合的语法坑**：在 Pandas 里组合多个条件时，**不能用 `and` / `or`**，必须改用符号 **`&`**（并且）、**`|`**（或者）、**`~`**（取反）；而且由于运算符优先级的原因，**每一个条件都必须用圆括号 `(...)` 括起来**！

例如，我们要筛选出“含有至少 3 个芳香环，且完全不含杂原子”的多环芳烃（PAH）：

In [ ]:
pah = df[(df["NumAromaticRings"] >= 3) & (df["NumHeteroatoms"] == 0)]
print(f"共筛出 {len(pah)} 个多环芳烃：")
pah[["Name", "SMILES", "Solubility", "NumAromaticRings"]].head(8)

如果想按字符串列（比如 `Name` 或 `SMILES`）来筛选，可以通过 `.str` 调用各种字符串方法，比如 `.str.lower()`（转小写）、`.str.isin(...)`（在名单内）或 `.str.contains(...)`（包含某个词）：

In [ ]:
# 找出名字里有 "thiophene"（噻吩，不区分大小写）的化合物
df[df["Name"].str.contains("thiophene", case=False, na=False)][["Name", "SMILES", "Solubility"]].head(8)

### 🏋️ 练习 01b-2

1. 模仿 4.2 节 `.idxmin()` 的用法，用 `df["MolWt"].idxmax()` 找出数据集中**相对分子质量最大**的那个化合物，打印出它的 `Name`、`MolWt` 和 `Solubility`。
2. 统计一下数据集中同时满足 **`MolLogP > 5` 且 `Solubility < -6`**（既高度疏水、又极难溶）的化合物一共有多少个？

In [ ]:
# 在这里写你的代码


---
## 5. 排序与极值查找

除了按条件过滤，我们经常想把筛选出来的数据按某个指标排个序。用 `df.sort_values("列名")` 默认按从小到大（升序）排列，加上 `ascending=False` 则按从大到小（降序）排列。

下面我们把常见稠环芳烃（苯、萘、蒽、芘、苝）从表里挑出来，按芳香环数从小到大排个队，看看溶解度是怎么随共轭核增大而变化的：

In [ ]:
acene_names = ["benzene", "naphthalene", "anthracene", "pyrene", "perylene"]
acenes_df = df[df["Name"].str.lower().isin(acene_names)]

acenes_df[["Name", "NumAromaticRings", "MolWt", "Solubility"]].sort_values("MolWt")

可以看到非常清晰的阶梯式下降：单环的苯 $\log S = -1.64$，二环的萘降到 $-3.61$，三环的蒽降到 $-6.59$，五环的苝（perylene）更是跌到了 $-8.80$！因为 $\log S$ 是以 10 为底的对数，从苯到苝，$\log S$ 差了约 $7.16$ 个单位——意味着水溶解度足足下降了 **$10^7$（一千万）倍**以上。

如果只想看整张表里最难溶或最易溶的前几名，也可以直接用 `sort_values`（或者快捷方法 `df.nsmallest(5, "Solubility")` / `df.nlargest(5, "Solubility")`）：

In [ ]:
# 看看全库里最难溶于水的前 5 个化合物（升序前 5 行）：
df.sort_values("Solubility")[["Name", "MolWt", "MolLogP", "Solubility"]].head(5)

---
## 6. 特征计算与分组统计

### 6.1 整列运算新增一列

因为 Pandas 的列底层是 NumPy 数组，所以直接对整列做加减乘除或指数运算，就能生成新的特征列。比如把对数溶解度 $\log S$ 还原成摩尔浓度（mol/L），再乘以分子量换算成质量浓度（g/L）：

In [ ]:
df["Solubility_molL"] = 10 ** df["Solubility"]                  # mol/L
df["Solubility_gL"] = df["Solubility_molL"] * df["MolWt"]       # g/L

df[["Name", "Solubility", "Solubility_molL", "Solubility_gL"]].head()

### 6.2 用 `.apply()` 对每个元素应用自定义函数

如果新列的逻辑包含 `if / elif / else` 判断（或者后面 `01d` 要讲的 RDKit 分子操作），没法直接写成简单的加减乘除，就可以先写一个普通的 Python 函数，再用 `.apply(函数名)` 让 Pandas 自动把这一列的每个值挨个传进去计算：

In [ ]:
def solubility_class(logs):
    if logs >= 0:
        return "highly soluble"     # >= 1 mol/L
    elif logs >= -2:
        return "soluble"            # 0.01 ~ 1 mol/L
    elif logs >= -4:
        return "slightly soluble"   # 0.1 mmol/L ~ 0.01 mol/L
    else:
        return "insoluble"          # < 0.1 mmol/L

df["sol_class"] = df["Solubility"].apply(solubility_class)
df["sol_class"].value_counts()

### 6.3 分组统计（`groupby`）

刚才我们只看了苯、萘、蒽、芘、苝这 5 个分子，就猜想“芳香环越多越难溶”。在全部 9482 个分子上，这个规律还成立吗？

用 `df.groupby("分组列")["统计列"].agg([...])` 可以按某个类别或整数把数据分成若干组，分别算出每组的均值、样本数等统计量（相当于 Excel 里的数据透视表）：

In [ ]:
by_rings = df.groupby("NumAromaticRings")["Solubility"].agg(["mean", "std", "count"])
by_rings.head(8).round(2)

从 `0` 个芳香环到 `5` 个芳香环，平均 $\log S$ 从 `-2.25` 一路单调下降到 `-6.42`，规律非常扎实。但同时也要留意右边的 `count`（样本数）列：含有 6 个以上芳香环的分子在库里只有寥寥十几个甚至几个——**样本量太小时，算出来的均值极易受个别极端分子影响，统计上不再可靠**。看平均值时永远要瞄一眼样本数。

### 🏋️ 练习 01b-3

1. 按刚才生成的溶解度等级 `sol_class` 分组，计算每一组的**平均相对分子质量（`MolWt`）**和**平均疏水常数（`MolLogP`）**。
2. 观察算出来的表格，越难溶的组在分子量和疏水性上呈现什么趋势？

In [ ]:
# 在这里写你的代码


---
## 7. 多表合并、缺失值处理与保存

### 7.1 横向合并（`merge`）与缺失值（`NaN`）

研究工作中常遇到数据分散在两张表里的情况：比如你自己在实验本上整理了一张“常用芳烃在有机电子中的用途表”（`roles`），现在想把 AqSolDB 里的溶解度和分子量匹配过来。只要两张表有共同的键（比如统一转成小写的分子名 `name`），就可以用 `.merge()` 把它们横向拼合：

In [ ]:
roles = pd.DataFrame({
    "name": ["naphthalene", "anthracene", "pyrene", "perylene", "pentacene"],
    "role": ["building block", "early OFET", "fluorescent probe", "acceptor core", "OFET benchmark"],
})

df["name_lower"] = df["Name"].str.lower()
merged = roles.merge(df[["name_lower", "Solubility", "MolWt"]],
                     left_on="name", right_on="name_lower", how="left")
merged

注意最后一行 `pentacene`（并五苯）：它的 `Solubility` 和 `MolWt` 显示为 **`NaN`**（Not a Number，代表**缺失值**）。这是因为并五苯实在太难溶了，AqSolDB 数据库里根本没有收录它的水溶解度！

我们在 `merge` 时指定了 `how="left"`（左连接），意思是“以左表 `roles` 为准，全部保留；右表里找不到的就填 `NaN`”。如果改成 `how="inner"`（内连接），则只保留两张表里都存在的分子。

面对表格里的缺失值 `NaN`，常用的检查和清理方法如下：

In [ ]:
print("每列的缺失值个数:\n", merged.isna().sum())

# 最稳妥的做法：直接剔除含有缺失值的行
clean_merged = merged.dropna()
clean_merged

> Pandas 也提供 `.fillna(值)` 来填补缺失值，但在科学数据建模中要格外小心：**千万不要随便编造一个数字（比如 0 或 -99）塞进去当成真实实验值**，否则会严重误导机器学习模型。除非有明确的物理依据，通常优先剔除缺失样本，或者在特征列上使用训练集均值插补（第 2 章会具体讲）。

### 7.2 纵向拼接（`concat`）与保存结果

如果两张表的列结构完全一样（比如分别挑出了最易溶的 3 个分子和最难溶的 3 个分子），想把它们上下首尾接成一张表，用 `pd.concat([...])`：

In [ ]:
top3 = df.nlargest(3, "Solubility")[["Name", "Solubility", "MolWt"]]
bottom3 = df.nsmallest(3, "Solubility")[["Name", "Solubility", "MolWt"]]

extremes = pd.concat([top3, bottom3], ignore_index=True)
extremes

整理完的数据表，用 `.to_csv("文件名.csv", index=False)` 即可存盘。记得加上 `index=False`，否则 Pandas 会把左侧的 `0, 1, 2...` 行号也当成一列写进文件，下次读回来时就会多出一列讨厌的 `Unnamed: 0`：

In [ ]:
aromatic_df = df[df["NumAromaticRings"] >= 1]
aromatic_df.to_csv("aromatic_only.csv", index=False)
print(f"已将 {len(aromatic_df)} 个含芳香环的分子保存至 aromatic_only.csv")

---
## 小结

这一节我们用 Pandas 走完了处理科研表格数据的标准流程：
- **读入与体检**：`pd.read_csv()` 读表，第一时间用 `head()`、`shape`、`info()`、`describe()` 摸清数据规模与分布（还顺手抓出了分子量高达 5000+ 的异常大分子）；
- **定位与筛选**：单列 `df["col"]`、多列 `df[["a", "b"]]`，按位置用 `iloc`、按标签用 `loc`，按条件筛选用 `df[(条件1) & (条件2)]`；
- **变换与聚合**：通过向量化运算或 `.apply(函数)` 生成新特征列，用 `sort_values()` 排序，用 `groupby()` 分组统计验证了“芳香环越多、水溶解度越低”的规律；
- **拼接与导出**：用 `merge()` 横向关联、处理 `NaN` 缺失值，最后用 `to_csv(index=False)` 保存结果。

**下一节（`01c_plotting.ipynb`）**：光看统计数字还不够直观，我们将用 `matplotlib` 和 `seaborn` 把这近万个分子的分布、趋势和相关性画成一目了然的科研图表。